In [1]:
!pip install rank-bm25

In [2]:
# General Imports
import os
import torch
import numpy as np
import pickle
import copy

import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.nn.utils import spectral_norm
from torch.utils.data import Dataset, DataLoader
from accelerate import Accelerator
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

from rank_bm25 import BM25Okapi

In [3]:
AUG_FACTOR  = 5
NOISE_STD   = 0.1
GEN_LEARNING_RATE = 2e-5
DISC_LEARNING_RATE = 8e-5
BATCH_SIZE = 32
EPOCHS = 300
RNG = np.random.default_rng(42)

In [4]:
class GeneratorResidualBlock(nn.Module):
    def __init__(self, in_dim, out_dim, use_projection=False):
        super().__init__()
        self.use_projection = use_projection
        self.dense = nn.Linear(in_dim, out_dim)
        self.norm = nn.LayerNorm(out_dim)
        self.act = nn.LeakyReLU(0.2)
        
        if self.use_projection:
            self.skip = nn.Linear(in_dim, out_dim)
            
    def forward(self, x):
        identity = self.skip(x) if self.use_projection else x
        out = self.act(self.norm(self.dense(x)))
        return out + identity

class GANQueryExpanderGenerator(nn.Module):
    def __init__(self):
        super().__init__()
        # Input: 150 (query) + 64 (noise) + 150 (prf) = 364
        
        # self.block1 = GeneratorResidualBlock(364, 512, use_projection=True)
        # self.block2 = GeneratorResidualBlock(512, 512, use_projection=False)
        
        # # Output stage mapping to LSI space
        # self.out_dense1 = nn.Linear(512, 256)
        # self.out_norm = nn.LayerNorm(256)
        # self.out_act = nn.LeakyReLU(0.2)
        
        # self.out_dense2 = nn.Linear(256, 150)
        # self.tanh = nn.Tanh()

        self.block1 = GeneratorResidualBlock(364, 256, use_projection=True)
        self.block2 = GeneratorResidualBlock(256, 256, use_projection=False)
        
        # Output stage mapping to LSI space
        self.out_dense1 = nn.Linear(256, 150)
        self.out_norm = nn.LayerNorm(150)
        self.out_act = nn.LeakyReLU(0.2)
        
        self.out_dense2 = nn.Linear(150, 150)
        self.tanh = nn.Tanh()
        
    def forward(self, query_vec, noise, prf_ctx):
        # Concatenate [q || z || prf] -> 364-dim
        x = torch.cat([query_vec, noise, prf_ctx], dim=-1)
        x = self.block1(x)
        x = self.block2(x)
        
        x = self.out_act(self.out_norm(self.out_dense1(x)))
        # Final expanded query vector
        return self.tanh(self.out_dense2(x))

class GANQueryExpanderDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        # Input: 150 (candidate) + 150 (prf) = 300
        
        # self.net = nn.Sequential(
        #     nn.Linear(300, 256),
        #     nn.LayerNorm(256),
        #     nn.LeakyReLU(0.2),
        #     nn.Dropout(0.3),
            
        #     nn.Linear(256, 128),
        #     nn.LeakyReLU(0.2),
        #     nn.Dropout(0.3),
            
        #     # Linear output for WGAN-GP critic score
        #     nn.Linear(128, 1)
        # )

        self.net = nn.Sequential(
            nn.Linear(300, 128),
            nn.LayerNorm(128),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            
            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.LeakyReLU(0.2),
            
            # Linear output for WGAN-GP critic score
            spectral_norm(nn.Linear(64, 1))
        )
        
    def forward(self, candidate_vec, prf_ctx):
        x = torch.cat([candidate_vec, prf_ctx], dim=-1)
        return self.net(x)

In [5]:
def weights_init(m):
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, 0.0, 0.02)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

In [6]:
class CISITripleDataset(Dataset):
    def __init__(self, queries, prf_contexts, targets):
        """
        All inputs should be L2-normalized 256-dim tensors derived from TruncatedSVD.
        """
        self.queries = queries
        self.prf_contexts = prf_contexts
        self.targets = targets
        
    def __len__(self):
        return len(self.queries)
        
    def __getitem__(self, idx):
        return self.queries[idx], self.prf_contexts[idx], self.targets[idx]

def compute_gradient_penalty(discriminator, real_samples, fake_samples, prf_ctx, device):
    """Calculates the gradient penalty for WGAN-GP"""
    alpha = torch.rand((real_samples.size(0), 1), device=device)
    # Get random interpolation between real and fake samples
    interpolates = (alpha * real_samples + ((1 - alpha) * fake_samples)).requires_grad_(True)
    
    d_interpolates = discriminator(interpolates, prf_ctx)
    fake = torch.ones((real_samples.size(0), 1), device=device)
    
    # Get gradients with respect to the interpolations
    gradients = torch.autograd.grad(
        outputs=d_interpolates,
        inputs=interpolates,
        grad_outputs=fake,
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]
    
    gradients = gradients.view(gradients.size(0), -1)
    gradient_penalty = ((gradients.norm(2, dim=1) - 1) ** 2).mean()
    return gradient_penalty

def train_gan_qe(generator, discriminator, opt_G, opt_D, dataloader, accelerator, prefix, epochs, lambda_gp=10, n_critic=5):
    """
    Executes the underlying WGAN-GP training steps using components 
    already prepared by Accelerator.
    """

    generator.apply(weights_init)
    discriminator.apply(weights_init)
    
    generator.train()
    discriminator.train()

    ema_generator = copy.deepcopy(generator)
    EMA_DECAY = 0.999   # β=0.999 is the standard for small models

    best_g_loss = float('inf')
    
    for epoch in range(epochs):
        for i, (queries, prf_ctx, real_targets) in enumerate(dataloader):
            batch_sz = queries.size(0)
            device = accelerator.device
            # ==========================================
            # Train Discriminator (Critic) - 5 steps
            # ==========================================
            opt_D.zero_grad()
            
            # Generate fake expansion
            noise = torch.randn(batch_sz, 64, device=accelerator.device)
            fake_expansions = generator(queries, noise, prf_ctx)
            
            # Critic scores
            real_validity = discriminator(real_targets, prf_ctx)
            fake_validity = discriminator(fake_expansions.detach(), prf_ctx)
            
            # Gradient penalty
            gradient_penalty = compute_gradient_penalty(
                discriminator, real_targets, fake_expansions.detach(), prf_ctx, accelerator.device
            )
            
            # Adversarial loss
            d_loss = -torch.mean(real_validity) + torch.mean(fake_validity) + lambda_gp * gradient_penalty
            
            accelerator.backward(d_loss)
            opt_D.step()
            
            # ==========================================
            # 5 critic (D) steps
            # ==========================================
            for _ in range(n_critic):
                opt_D.zero_grad()
                noise_d         = torch.randn(queries.size(0), 64, device=device)
                with torch.no_grad():
                    fake_exp    = generator(queries, noise_d, prf_ctx)
                real_score      = discriminator(real_targets, prf_ctx)
                fake_score      = discriminator(fake_exp.detach(), prf_ctx)
                gp              = compute_gradient_penalty(
                                    discriminator, real_targets, fake_exp, prf_ctx, device)
                d_loss          = -torch.mean(real_score) + torch.mean(fake_score) + lambda_gp * gp
                accelerator.backward(d_loss)
                opt_D.step()

            # ==========================================
            # 1 generator (G) step
            # ==========================================
            opt_G.zero_grad()
            noise_g             = torch.randn(queries.size(0), 64, device=device)  # fresh noise!
            fake_exp            = generator(queries, noise_g, prf_ctx)
            fake_score          = discriminator(fake_exp, prf_ctx)
            g_loss              = -torch.mean(fake_score)
            accelerator.backward(g_loss)
            opt_G.step()

            with torch.no_grad():
                for ema_p, g_p in zip(ema_generator.parameters(), generator.parameters()):
                    ema_p.data.mul_(EMA_DECAY).add_(g_p.data, alpha=1 - EMA_DECAY)

        current_g_loss = g_loss.item()
        if current_g_loss < best_g_loss:
            best_g_loss = current_g_loss
            
            if accelerator.is_local_main_process:
                # Overwrite the previous best file to save space
                torch.save(
                    accelerator.unwrap_model(generator).state_dict(),
                    f"{prefix}_model_best.pt"
                )
                
                torch.save(
                    accelerator.unwrap_model(ema_generator).state_dict(),
                    f"{prefix}_model_ema.pt"      
                )
                
        if accelerator.is_local_main_process and (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1:03d}/{epochs} | D Loss: {d_loss.item():.4f} | G Loss: {current_g_loss:.4f} | Best G: {best_g_loss:.4f}")

    # # Save model safely using accelerator
    # if accelerator.is_local_main_process:
    #     accelerator.save_model(generator, "kaggle_working/generator")
    #     print("Training complete. Generator saved.")
    if accelerator.is_local_main_process:
        accelerator.save_model(generator, "kaggle_working/generator")
        # Save final EMA weights as a standalone .pt file too
        torch.save(
            accelerator.unwrap_model(ema_generator).state_dict(),
            f"{prefix}_model_ema_final.pt"
        )
        print("Training complete. Generator and EMA saved.")
        
    return generator

In [7]:
# Assume train_gan_qe and CISITripleDataset are imported from our previous script
# from gan_pipeline import train_gan_qe, CISITripleDataset

def build_canonical_space(docs):
    """
    Builds the fixed canonical LSI space: log-TF x IDF + L2 norm + SVD(150)
    """
    # 1. Canonical TF-IDF: sublinear_tf=True applies log(TF), norm='l2' applies L2
    vectorizer = TfidfVectorizer(
        sublinear_tf=True, 
        use_idf=True,      
        norm='l2',
        max_features = 5000,
        token_pattern = r'(?u)\b[a-zA-Z]{3,}\b',
        min_df=4,           # Exclude terms appearing in fewer than 4 documents
        max_df=0.80         # Exclude terms appearing in more than 80% of documents
    )
    tfidf_matrix = vectorizer.fit_transform(docs)
    
    # 2. LSI Projection
    svd = TruncatedSVD(n_components=150, random_state=42)
    doc_lsi = svd.fit_transform(tfidf_matrix)
    
    # 3. L2 Normalize everything to ensure only direction matters
    doc_vectors = normalize(doc_lsi, norm='l2')
    vocab_vectors = normalize(svd.components_.T, norm='l2')
    
    return vectorizer, svd, doc_vectors, vocab_vectors

def prepare_training_triples(queries_dict, qrels_dict, corpus_dict, vectorizer, svd, doc_vectors, bm25):
    """
    Generates (query, PRF context, target) triples for WGAN-GP training.
    Safely aligns string/integer keys and maps 1-based IDs to 0-based matrix row indices.
    """
    q_vecs, prf_vecs, target_vecs = [], [], []
    
    # 1. Create a foolproof map from document ID (string or int) to its index in doc_vectors
    # Since preprocessed_docs was built from corpus_dict.values(), this matches perfectly.
    doc_id_to_idx = {str(doc_id): idx for idx, doc_id in enumerate(corpus_dict.keys())}
    
    # 2. Normalize qrels keys to strings so they match queries_dict keys
    normalized_qrels = {str(k): [str(v_id) for v_id in v_list] for k, v_list in qrels_dict.items()}
    
    for qid, q_text in queries_dict.items():
        str_qid = str(qid)
        
        # Fix the type mismatch check
        if str_qid not in normalized_qrels:
            continue
            
        # Embed query into LSI space
        q_tfidf = vectorizer.transform([q_text])
        q_lsi = normalize(svd.transform(q_tfidf), norm='l2')[0]
        
        # 3. Map ground truth relevant doc IDs to their actual 0-based matrix row indices
        relevant_ids = normalized_qrels[str_qid]
        relevant_indices = [doc_id_to_idx[d_id] for d_id in relevant_ids if d_id in doc_id_to_idx]
        
        if not relevant_indices: 
            continue
            
        # Target Vector: Mean of truly relevant document vectors
        target_vec = np.mean(doc_vectors[relevant_indices], axis=0)
        target_vec = normalize(target_vec.reshape(1, -1), norm='l2')[0]
        
        # PRF Context: Top-5 document indices based on LSI cosine similarity
        # sims = np.dot(doc_vectors, q_lsi)
        # top_5_idx = np.argsort(sims)[::-1][:5]
        bm25_scores = bm25.get_scores(q_text.split())
        top_5_idx   = np.argsort(bm25_scores)[::-1][:5]
        prf_vec = np.mean(doc_vectors[top_5_idx], axis=0)
        prf_vec = normalize(prf_vec.reshape(1, -1), norm='l2')[0]
        
        q_vecs.append(q_lsi)
        prf_vecs.append(prf_vec)
        target_vecs.append(target_vec)

        individual_prf_vecs = [doc_vectors[i] for i in top_5_idx[:AUG_FACTOR]]
        for pv in individual_prf_vecs:
            noise_t   = RNG.normal(0, NOISE_STD, size=target_vec.shape)
            aug_target = normalize((target_vec + noise_t).reshape(1,-1), norm='l2')[0]
            q_vecs.append(q_lsi)
            prf_vecs.append(normalize(pv.reshape(1,-1), norm='l2')[0])
            target_vecs.append(aug_target)
        
    print(f"Successfully generated {len(q_vecs)} training triples.")
    
    # Convert lists to NumPy arrays first to prevent PyTorch tensor warnings
    return torch.tensor(np.array(q_vecs), dtype=torch.float32), \
           torch.tensor(np.array(prf_vecs), dtype=torch.float32), \
           torch.tensor(np.array(target_vecs), dtype=torch.float32)

In [8]:
class QueryExpander:
    def __init__(self, use_stemming: bool, use_stopwords: bool, model_dir="pretrained_qegans"):
        """
        Loads the pre-computed canonical space and GAN generator for the specific configuration.
        """
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        prefix = f"{model_dir}/stem_{use_stemming}_stop_{use_stopwords}"
        
        # Load Canonical Space (Vectorizers + SVD)
        with open(f"{prefix}_space.pkl", "rb") as f:
            space = pickle.load(f)
            self.vectorizer = space['vectorizer']
            self.svd = space['svd']
            self.doc_vectors = space['doc_vectors']
            self.vocab_vectors = space['vocab_vectors']
            self.vocab_terms = self.vectorizer.get_feature_names_out()
            
        # Load Generator
        self.generator = GANQueryExpanderGenerator().to(self.device)
        self.generator.load_state_dict(torch.load(f"{prefix}_model_best.pt", map_location=self.device))
        self.generator.load_state_dict(torch.load(f"{prefix}_model_ema.pt", map_location=self.device))
        self.generator.eval()

    def expand(self, preprocessed_query_str: str, top_k: int = 5, return_all: bool = False):
        """
        Takes the user's query, projects it into the canonical LSI space, 
        generates the expansion vector, and decodes it back to terms.
        """
        # 1. Project query into canonical LSI space
        q_tfidf = self.vectorizer.transform([preprocessed_query_str])
        q_lsi = self.svd.transform(q_tfidf)
        q_norm = q_lsi / (np.linalg.norm(q_lsi) + 1e-8)
        
        # 2. Get PRF Context (Using canonical doc vectors)
        sims = np.dot(self.doc_vectors, q_norm[0])
        top_5_idx = np.argsort(sims)[::-1][:5]
        prf_ctx = np.mean(self.doc_vectors[top_5_idx], axis=0)
        prf_norm = prf_ctx / (np.linalg.norm(prf_ctx) + 1e-8)
        
        # 3. Generate Expansion via GAN
        q_tensor = torch.tensor(q_norm[0], dtype=torch.float32).to(self.device)
        prf_tensor = torch.tensor(prf_norm, dtype=torch.float32).to(self.device)
        noise = torch.zeros(64, dtype=torch.float32).to(self.device) # Deterministic at inference
        
        with torch.no_grad():
            # Generator output is already in 256-dim LSI space
            expanded_vec = self.generator(q_tensor, noise, prf_tensor).cpu().numpy()
            
        # 4. Decode via Cosine Similarity in canonical space
        e_norm = expanded_vec / (np.linalg.norm(expanded_vec) + 1e-8)
        
        # Because vocab_vectors are already L2 normalized, dot product == cosine similarity
        scores = np.dot(self.vocab_vectors, e_norm) 

        query_tokens = preprocessed_query_str.split()
        for i, term in enumerate(self.vocab_terms):
            if term in query_tokens:
                scores[i] = -9999.0  # Force original terms to the bottom of the rank
        
        if return_all:
            top_indices = np.argsort(scores)[::-1]
            return [(self.vocab_terms[i], float(scores[i])) for i in top_indices]
        else:
            top_indices = np.argsort(scores)[::-1][:top_k]
            return [(self.vocab_terms[i], float(scores[i])) for i in top_indices]

In [9]:
import re
import nltk
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

_stemmer = PorterStemmer()
_stopwords = None

# Returns the cached English stopword set, downloading the NLTK corpus on first use
def _get_stopwords() -> set[str]:
    global _stopwords
    if _stopwords is None:
        try:
            _stopwords = set(stopwords.words("english"))
        except LookupError:
            nltk.download("stopwords")
            _stopwords = set(stopwords.words("english"))
    return _stopwords

# Parses the cisi.all or query.text file into { id: text }, concatenating the given fields.
# CISI documents (cisi.all) use .T (title) and .W (text); queries (query.text) use .W.
def load_smart(path: str, fields: tuple[str, ...] = ("T", "W")) -> dict[str, str]:
    records: dict[str, list[str]] = {}
    current_id = None
    current_field = None

    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.startswith(".I"):
                current_id = line[2:].strip()
                current_field = None
                records[current_id] = []
            elif line.startswith("."):
                current_field = line[1:2]
            elif current_id is not None and current_field in fields:
                records[current_id].append(line.strip())

    return {doc_id: " ".join(parts) for doc_id, parts in records.items()}

# Tokenizes one string into a list of terms.
def preprocess(text: str, stem: bool = True, remove_stopwords: bool = True) -> list[str]:
    tokens = re.findall(r"[a-z]+", text.lower())

    if remove_stopwords:
        sw = _get_stopwords()
        tokens = [t for t in tokens if t not in sw]

    if stem:
        tokens = [_stemmer.stem(t) for t in tokens]

    return tokens

# Applies preprocess() across a { id: text } collection.
def preprocess_collection(
    docs: dict[str, str], stem: bool = True, remove_stopwords: bool = True
) -> dict[str, list[str]]:
    return {
        doc_id: preprocess(text, stem=stem, remove_stopwords=remove_stopwords)
        for doc_id, text in docs.items()
    }


In [10]:
def pretrain_all_conditions(corpus_dict, queries_dict, qrels_dict):
    accelerator = Accelerator(mixed_precision="fp16") # T4 GPUs excel with fp16
    
    conditions = [
        (True, True),   # Stemming=True, Stopwords=True
        (True, False),  # Stemming=True, Stopwords=False
        (False, True),  # Stemming=False, Stopwords=True
        (False, False)  # Stemming=False, Stopwords=False
    ]
    
    # Only the main process creates the directory
    if accelerator.is_local_main_process:
        os.makedirs("pretrained_qegans", exist_ok=True)
    
    for stem, stop in conditions:
        if accelerator.is_local_main_process:
            print(f"\n--- Training Condition: Stemming={stem}, Stopwords={stop} ---")

        # Apply Stemming/Stopword Condition
        preprocessed_docs = [
            " ".join(preprocess(text, stem=stem, remove_stopwords=stop)) 
            for text in corpus_dict.values()
        ]
        
        preprocessed_queries = {
            qid: " ".join(preprocess(q, stem=stem, remove_stopwords=stop))
            for qid, q in queries_dict.items()
        }

        tokenized_corpus = [text.split() for text in preprocessed_docs]
        bm25 = BM25Okapi(tokenized_corpus)
            
        # Build Canonical Space
        vectorizer, svd, doc_vectors, vocab_vectors = build_canonical_space(preprocessed_docs)
        
        # Build Dataset
        q_t, prf_t, target_t = prepare_training_triples(
            preprocessed_queries, qrels_dict, corpus_dict, vectorizer, svd, doc_vectors, bm25
        )
        
        dataset = CISITripleDataset(q_t, prf_t, target_t)
        
        # 1. Initialize fresh models for this condition
        generator = GANQueryExpanderGenerator()
        discriminator = GANQueryExpanderDiscriminator()
        
        opt_G = optim.Adam(generator.parameters(), lr=GEN_LEARNING_RATE, betas=(0.5, 0.9))
        opt_D = optim.Adam(discriminator.parameters(), lr=DISC_LEARNING_RATE, betas=(0.5, 0.9))
        
        dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
        
        # 2. Prepare everything for the current loop
        gen, disc, opt_g, opt_d, loader = accelerator.prepare(
            generator, discriminator, opt_G, opt_D, dataloader
        )

        prefix = f"pretrained_qegans/stem_{stem}_stop_{stop}"
        
        #4. Train Model
        # Run execution loop passing target DDP containers
        train_gan_qe(gen, disc, opt_g, opt_d, loader, accelerator, prefix, epochs=EPOCHS)
        
        # CRITICAL: Force synchronization across both T4 instances before hitting disk IO
        accelerator.wait_for_everyone() 
        
        # Unwrap the generator model to strip DDP hooks ('module.') before saving
        unwrapped_gen = accelerator.unwrap_model(gen)
        
        # Isolate saving strictly to the main process
        if accelerator.is_local_main_process:
            # torch.save(unwrapped_gen.state_dict(), f"{prefix}_model.pt")
            
            # Cache coordinate parameters & mapping states safely
            with open(f"{prefix}_space.pkl", "wb") as f:
                pickle.dump({
                    'vectorizer': vectorizer,
                    'svd': svd,
                    'doc_vectors': doc_vectors,
                    'vocab_vectors': vocab_vectors
                }, f)
            print(f"Successfully serialized weights and vectors to {prefix}")

In [ ]:
def load_qrels(filepath):
    qrels = {}
    with open(filepath, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 2: continue
            q_id   = int(parts[0])
            d_id   = int(parts[1])
            if q_id not in qrels:
                qrels[q_id] = []
            qrels[q_id].append(d_id)
    return qrels

In [ ]:
# 1. Define Kaggle input paths
# Replace 'cisi-ir-dataset' with whatever you named the dataset during upload
DATA_DIR = "/kaggle/input/datasets/farhannr28/cisi-ir-dataset"

docs_path = os.path.join(DATA_DIR, "cisi.all")
queries_path = os.path.join(DATA_DIR, "query.text")
qrels_path = os.path.join(DATA_DIR, "qrels.text")

# 2. Parse the files
print("Parsing CISI dataset...")
corpus = load_smart(docs_path, fields=("T", "W", "A"))
queries = load_smart(queries_path, fields=("W",))
qrels = load_qrels(qrels_path)

print(f"Loaded {len(corpus)} documents, {len(queries)} queries, and {len(qrels)} relevance judgments.")

# 3. Fire off the distributed multi-GPU training
# This will loop through all 4 stem/stopword conditions and save the models
pretrain_all_conditions(corpus, queries, qrels)

Parsing CISI dataset...
Loaded 1460 documents, 112 queries, and 76 relevance judgments.

--- Training Condition: Stemming=True, Stopwords=True ---
Successfully generated 456 training triples.
Epoch 010/300 | D Loss: -0.1047 | G Loss: -0.4633 | Best G: -0.6138
Epoch 020/300 | D Loss: -0.1329 | G Loss: -0.5762 | Best G: -0.6138
Epoch 030/300 | D Loss: 0.0342 | G Loss: -0.3050 | Best G: -0.6138
Epoch 040/300 | D Loss: -0.1815 | G Loss: -0.4256 | Best G: -0.6138
Epoch 050/300 | D Loss: -0.1087 | G Loss: -0.6721 | Best G: -0.7206
Epoch 060/300 | D Loss: -0.3014 | G Loss: -0.8228 | Best G: -0.8228
Epoch 070/300 | D Loss: -0.1656 | G Loss: -0.7661 | Best G: -1.0956
Epoch 080/300 | D Loss: -0.1608 | G Loss: -1.2433 | Best G: -1.2433
Epoch 090/300 | D Loss: -0.2801 | G Loss: -1.2087 | Best G: -1.4198
Epoch 100/300 | D Loss: -0.3070 | G Loss: -1.2816 | Best G: -1.4198
Epoch 110/300 | D Loss: -0.1230 | G Loss: -1.4705 | Best G: -1.5463
Epoch 120/300 | D Loss: -0.4873 | G Loss: -1.3681 | Best G: -

In [13]:
def human_audit_pipeline(raw_query: str, top_k: int = 10, model_dir="pretrained_qegans"):
    """
    Feeds a test query into all 4 trained conditions to let a human 
    judge the semantic quality of the expansions.
    """
    print(f"\n" + "="*60)
    print(f"RAW USER QUERY: '{raw_query}'")
    print("="*60)
    
    conditions = [
        (True, True, "Stemming=ON,  Stopwords=REMOVED"),
        (True, False, "Stemming=ON,  Stopwords=KEPT"),
        (False, True, "Stemming=OFF, Stopwords=REMOVED"),
        (False, False, "Stemming=OFF, Stopwords=KEPT")
    ]
    
    for stem, stop, label in conditions:
        print(f"\n▶ CONFIGURATION: {label}")
        
        # 1. Apply matching preprocessing to the raw query string
        tokens = preprocess(raw_query, stem=stem, remove_stopwords=stop)
        preprocessed_query_str = " ".join(tokens)
        print(f"   Preprocessed Input: '{preprocessed_query_str}'")
        
        # 2. Check if the model files exist before loading
        prefix = f"{model_dir}/stem_{stem}_stop_{stop}"
        if not os.path.exists(f"{prefix}_model_best.pt") or not os.path.exists(f"{prefix}_space.pkl"):
            print("   [Error] Model or Space file not found for this configuration. Skipping.")
            continue
            
        try:
            # 3. Instantiate the expander (loads the best weights & space vectors)
            expander = QueryExpander(use_stemming=stem, use_stopwords=stop, model_dir=model_dir)
            
            # 4. Generate the expansion terms
            expanded_pairs = expander.expand(preprocessed_query_str, top_k=top_k)
            
            # 5. Print a clean, formatted table of the results
            print(f"   Top-{top_k} Expanded Terms & Cosine Weights:")
            print(f"   {'-'*45}")
            for rank, (term, score) in enumerate(expanded_pairs, 1):
                # Highlight if the term was already in the original query
                is_original = " (original)" if term in tokens else ""
                print(f"     [{rank:02d}]  {term:<18} | Score: {score:.4f}{is_original}")
                
        except Exception as e:
            print(f"   [Error executing inference]: {str(e)}")

In [14]:
human_audit_pipeline("automated information retrieval systems", top_k=8)


RAW USER QUERY: 'automated information retrieval systems'

▶ CONFIGURATION: Stemming=ON,  Stopwords=REMOVED
   Preprocessed Input: 'autom inform retriev system'
   Top-8 Expanded Terms & Cosine Weights:
   ---------------------------------------------
     [01]  librari            | Score: 0.2634
     [02]  use                | Score: 0.2375
     [03]  remot              | Score: 0.2284
     [04]  result             | Score: 0.2268
     [05]  california         | Score: 0.2138
     [06]  test               | Score: 0.2131
     [07]  equip              | Score: 0.2115
     [08]  perform            | Score: 0.2099

▶ CONFIGURATION: Stemming=ON,  Stopwords=KEPT
   Preprocessed Input: 'autom inform retriev system'
   Top-8 Expanded Terms & Cosine Weights:
   ---------------------------------------------
     [01]  user               | Score: 0.2236
     [02]  for                | Score: 0.2162
     [03]  perform            | Score: 0.2145
     [04]  test               | Score: 0.2090
    